# CreditWise AI — Model Training and Evaluation

This phase compares Logistic Regression, Decision Tree, and Random Forest models under two feature scenarios: all 11 input features, and the same inputs without `loan_grade`. Each scenario is evaluated with the estimator's default class weighting and `class_weight="balanced"` (12 fitted pipelines total). The split is the persisted Phase 2 split; it is not recreated.

Every estimator is packaged with its own preprocessing pipeline. Imputation, scaling, and category encoding are fitted on training inputs only. The test set is used only for the final reported metrics and plots—not model selection, tuning, or threshold selection. No predictions here represent actual loan approvals or credit decisions.

## 1. Load and validate the saved Phase 2 split

Phase 2 saved the stratified partitions as combined `credit_risk_train.csv` and `credit_risk_test.csv`, not as separate `X_train.csv`, `y_train.csv`, etc. This notebook reads those existing artifacts and separates the target in memory. It verifies sample counts and target class counts against the preprocessing configuration before fitting any model.

In [ ]:
from pathlib import Path
import json
import sys
import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, average_precision_score, confusion_matrix,
    f1_score, precision_score, recall_score, roc_auc_score,
    precision_recall_curve, roc_curve,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["axes.titleweight"] = "bold"

project_root = next(
    (candidate for candidate in [Path.cwd(), *Path.cwd().parents]
     if (candidate / "data/processed/credit_risk_preprocessing_config.json").is_file()),
    None,
)
if project_root is None:
    raise FileNotFoundError("Could not find the Phase 2 preprocessing configuration from this notebook's working directory.")

processed_dir = project_root / "data/processed"
models_dir = project_root / "models"
reports_dir = project_root / "reports"
models_dir.mkdir(parents=True, exist_ok=True)
reports_dir.mkdir(parents=True, exist_ok=True)
config_path = processed_dir / "credit_risk_preprocessing_config.json"
config = json.loads(config_path.read_text(encoding="utf-8"))
train_path = project_root / config["split"]["train_file"]
test_path = project_root / config["split"]["test_file"]
cleaned_path = project_root / config["cleaned_file"]
for required_path in [train_path, test_path, cleaned_path]:
    if not required_path.is_file():
        raise FileNotFoundError(f"Required Phase 2 artifact is missing: {required_path}")

train_frame = pd.read_csv(train_path)
test_frame = pd.read_csv(test_path)
cleaned_frame = pd.read_csv(cleaned_path)
target_column = config["target_column"]
feature_columns = config["feature_columns"]
X_train = train_frame[feature_columns].copy()
y_train = train_frame[target_column].astype(int).copy()
X_test = test_frame[feature_columns].copy()
y_test = test_frame[target_column].astype(int).copy()

assert ".venv" in str(Path(sys.executable)).lower(), f"Expected project .venv; running {sys.executable}"
assert len(cleaned_frame) == config["cleaned_shape"]["rows"]
assert len(X_train) == config["split"]["train_rows"]
assert len(X_test) == config["split"]["test_rows"]
assert X_train.columns.tolist() == feature_columns == X_test.columns.tolist()
assert target_column not in X_train.columns
assert y_train.value_counts().sort_index().to_dict() == {int(k): v for k, v in config["split"]["train_target_counts"].items()}
assert y_test.value_counts().sort_index().to_dict() == {int(k): v for k, v in config["split"]["test_target_counts"].items()}
assert set(y_train.unique()) == {0, 1} and set(y_test.unique()) == {0, 1}

print(f"Python: {sys.version.split()[0]} | scikit-learn: {sklearn.__version__} | interpreter: {sys.executable}")
print(f"Cleaned dataset: {len(cleaned_frame):,} rows")
print(f"Reused train/test split: {len(X_train):,} / {len(X_test):,}; random_state={config['split']['random_state']}; stratified by {target_column}")
print("Training class counts:", y_train.value_counts().sort_index().to_dict())
print("Testing class counts:", y_test.value_counts().sort_index().to_dict())
print("All input columns:", feature_columns)
print("Separate X_train/X_test/y_train/y_test files were absent; using the persisted combined split CSVs without recreating the split.")

## 2. Define feature scenarios and model configurations

**Scenario A** retains all 11 approved Phase 2 features. **Scenario B** excludes only `loan_grade` (10 features). `loan_grade` is intentionally retained in Scenario A: it may encode the lender's original underwriting assessment, so results with and without it should be interpreted as a leakage/availability sensitivity comparison, not as evidence it is appropriate at prediction time.

The default and balanced class-weight settings are evaluated separately for all six algorithm/scenario combinations. Class weighting is not assumed to improve performance. Models use fixed settings and random states; no test-driven tuning or threshold adjustment is performed. Classification metrics and confusion matrices use the standard 0.50 probability threshold.

In [ ]:
numeric_features = config["numerical_features"]
categorical_features = config["categorical_features"]
scenarios = {
    "A_all_features": feature_columns.copy(),
    "B_without_loan_grade": [column for column in feature_columns if column != "loan_grade"],
}
assert len(scenarios["A_all_features"]) == 11
assert len(scenarios["B_without_loan_grade"]) == 10
assert "loan_grade" in scenarios["A_all_features"] and "loan_grade" not in scenarios["B_without_loan_grade"]

def make_preprocessor(columns):
    """Construct an independent preprocessing graph for one feature scenario."""
    numeric = [column for column in columns if column in numeric_features]
    categorical = [column for column in columns if column in categorical_features]
    return ColumnTransformer(transformers=[
        ("numeric", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]), numeric),
        ("categorical", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore")),
        ]), categorical),
    ], remainder="drop")

def make_estimators(class_weight):
    """Return fixed, reproducible classifier settings for the requested weight policy."""
    return {
        "Logistic_Regression": LogisticRegression(
            max_iter=1_000, class_weight=class_weight, random_state=42,
        ),
        "Decision_Tree": DecisionTreeClassifier(
            random_state=42, class_weight=class_weight,
        ),
        "Random_Forest": RandomForestClassifier(
            n_estimators=300, random_state=42, n_jobs=-1,
            class_weight=class_weight,
        ),
    }

results = []
fitted_pipelines = {}
predictions = {}
probabilities = {}
for scenario_name, columns in scenarios.items():
    scenario_X_train = X_train[columns]
    scenario_X_test = X_test[columns]
    for weight_name, class_weight in [("default", None), ("balanced", "balanced")]:
        for model_name, estimator in make_estimators(class_weight).items():
            run_name = f"{scenario_name}__{model_name}__{weight_name}"
            pipeline = Pipeline([
                ("preprocessor", make_preprocessor(columns)),
                ("classifier", estimator),
            ])
            with warnings.catch_warnings(record=True) as caught_warnings:
                warnings.simplefilter("always")
                pipeline.fit(scenario_X_train, y_train)
            y_pred = pipeline.predict(scenario_X_test)
            y_score = pipeline.predict_proba(scenario_X_test)[:, 1]
            metrics = {
                "configuration": run_name,
                "scenario": scenario_name,
                "model": model_name.replace("_", " "),
                "class_weight": weight_name,
                "feature_count": len(columns),
                "accuracy": accuracy_score(y_test, y_pred),
                "precision_default": precision_score(y_test, y_pred, zero_division=0),
                "recall_default": recall_score(y_test, y_pred, zero_division=0),
                "f1_default": f1_score(y_test, y_pred, zero_division=0),
                "roc_auc": roc_auc_score(y_test, y_score),
                "pr_auc_average_precision": average_precision_score(y_test, y_score),
                "confusion_matrix": confusion_matrix(y_test, y_pred, labels=[0, 1]).tolist(),
                "warnings": [str(item.message) for item in caught_warnings],
            }
            results.append(metrics)
            fitted_pipelines[run_name] = pipeline
            predictions[run_name] = y_pred
            probabilities[run_name] = y_score

results_df = pd.DataFrame(results)
assert len(results_df) == 12
assert results_df["configuration"].is_unique
display_columns = [
    "scenario", "model", "class_weight", "accuracy", "precision_default",
    "recall_default", "f1_default", "roc_auc", "pr_auc_average_precision",
]
display(results_df[display_columns].sort_values(
    ["scenario", "class_weight", "model"]
).style.format({column: "{:.3f}" for column in display_columns[3:]}))
print("All six algorithm/scenario pairs were fitted with both class-weight settings (12 pipelines).")
print("Recall is the share of actual defaults detected at the fixed 0.50 threshold; precision is the share of predicted defaults that were actual defaults.")

## 3. Evaluation visualizations

The following plots compare default detection, ranking quality, and trade-offs at the fixed classification threshold. ROC and precision-recall curves use predicted default probabilities. Confusion matrix axes show actual versus predicted classes; `1` is default.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7))
for ax, metric, title in [
    (axes[0], "recall_default", "Default Recall at Threshold 0.50"),
    (axes[1], "f1_default", "Default-Class F1 at Threshold 0.50"),
]:
    sns.barplot(data=results_df, x="model", y=metric, hue="scenario", errorbar=None, ax=ax)
    ax.set_title(title)
    ax.set_xlabel("Classifier")
    ax.set_ylabel(metric.replace("_", " ").title())
    ax.tick_params(axis="x", rotation=20)
    ax.set_ylim(0, 1)
    ax.legend(title="Feature scenario", fontsize=8)
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(2, 3, figsize=(17, 10))
for ax, (run_name, y_pred) in zip(axes.flat, predictions.items()):
    cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
    sns.heatmap(cm, annot=True, fmt=",d", cmap="Blues", cbar=False, ax=ax,
                xticklabels=["Non-default (0)", "Default (1)"],
                yticklabels=["Non-default (0)", "Default (1)"])
    result = results_df.loc[results_df["configuration"] == run_name].iloc[0]
    ax.set_title(f"{result['model']} | {result['scenario'].split('_', 1)[0]} | {result['class_weight']}")
    ax.set_xlabel("Predicted class")
    ax.set_ylabel("Actual class")
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 12))
for row_index, (scenario_name, columns) in enumerate(scenarios.items()):
    for run_name, scores in probabilities.items():
        if not run_name.startswith(scenario_name + "__"):
            continue
        result = results_df.loc[results_df["configuration"] == run_name].iloc[0]
        label = f"{result['model']} ({result['class_weight']}, AUC={result['roc_auc']:.3f})"
        fpr, tpr, _ = roc_curve(y_test, scores)
        axes[row_index, 0].plot(fpr, tpr, label=label)
        precision, recall, _ = precision_recall_curve(y_test, scores)
        axes[row_index, 1].plot(recall, precision, label=f"{result['model']} ({result['class_weight']}, AP={result['pr_auc_average_precision']:.3f})")
    axes[row_index, 0].plot([0, 1], [0, 1], linestyle="--", color="gray", label="Chance")
    axes[row_index, 0].set(title=f"ROC Curve — {scenario_name}", xlabel="False positive rate", ylabel="True positive rate")
    axes[row_index, 0].legend(fontsize=7, loc="lower right")
    axes[row_index, 1].axhline(y_test.mean(), linestyle="--", color="gray", label=f"Default prevalence={y_test.mean():.3f}")
    axes[row_index, 1].set(title=f"Precision–Recall Curve — {scenario_name}", xlabel="Recall (default)", ylabel="Precision (default)")
    axes[row_index, 1].legend(fontsize=7, loc="best")
plt.tight_layout()
plt.show()

In [ ]:
def feature_names_after_preprocessing(pipeline):
    """Return transformed feature labels in ColumnTransformer output order."""
    preprocessor = pipeline.named_steps["preprocessor"]
    return preprocessor.get_feature_names_out()

importance_runs = results_df.loc[
    results_df["model"].isin(["Decision Tree", "Random Forest"])
    & (results_df["class_weight"] == "default")
].sort_values(["scenario", "model"])
fig, axes = plt.subplots(len(importance_runs), 1, figsize=(12, 4 * len(importance_runs)))
if len(importance_runs) == 1:
    axes = [axes]
for ax, row in zip(axes, importance_runs.itertuples(index=False)):
    pipeline = fitted_pipelines[row.configuration]
    classifier = pipeline.named_steps["classifier"]
    names = feature_names_after_preprocessing(pipeline)
    importance = pd.Series(classifier.feature_importances_, index=names).sort_values().tail(15)
    ax.barh(importance.index, importance.values, color="#4C78A8")
    ax.set_title(f"Top 15 impurity-based importances — {row.model}, {row.scenario}")
    ax.set_xlabel("Feature importance")
plt.tight_layout()
plt.show()
print("Importance caveat: tree impurity importances describe this fitted model; they are not causal effects and can be biased toward features with many possible splits/categories.")

## 4. Interpretation and model-selection guidance

The text and comparisons below are computed from the held-out results; they do not choose thresholds or alter models. The strongest default detector is identified by recall. The best precision/recall balance is identified by F1, which is the harmonic mean of precision and recall. Interpret those results alongside PR-AUC, ROC-AUC, confusion counts, subgroup performance, and data availability—not accuracy alone.

Limitations: Logistic Regression is a linear model and may miss nonlinear relationships; class weighting can shift its operating trade-off. A single Decision Tree can overfit and is unstable to small data changes. Random Forest can capture nonlinear patterns but is less directly interpretable and may have higher compute cost. All models inherit sampling, historical underwriting, label, and potential proxy/fairness issues. `loan_grade` may be downstream of underwriting and suspiciously informative; performance with it should not be treated as a valid deployment estimate until availability and leakage are resolved.

In [ ]:
print("All 12 model results, ordered by default recall:")
display(results_df[display_columns].sort_values(
    ["recall_default", "f1_default", "pr_auc_average_precision"], ascending=False
).style.format({column: "{:.3f}" for column in display_columns[3:]}))

highest_recall = results_df.sort_values(
    ["recall_default", "pr_auc_average_precision", "precision_default"], ascending=False
).iloc[0]
best_f1 = results_df.sort_values(
    ["f1_default", "precision_default", "recall_default"], ascending=False
).iloc[0]
print(f"Highest default recall: {highest_recall['configuration']} — recall={highest_recall['recall_default']:.3f}, precision={highest_recall['precision_default']:.3f}, PR-AUC={highest_recall['pr_auc_average_precision']:.3f}.")
print(f"Highest default-class F1 (precision/recall balance): {best_f1['configuration']} — F1={best_f1['f1_default']:.3f}, precision={best_f1['precision_default']:.3f}, recall={best_f1['recall_default']:.3f}.")

comparison = results_df.pivot_table(
    index=["model", "class_weight"], columns="scenario",
    values=["accuracy", "precision_default", "recall_default", "f1_default", "roc_auc", "pr_auc_average_precision"],
)
print("Paired metric changes from Scenario A (all features) to Scenario B (without loan_grade); negative values mean lower without grade:")
for metric in ["recall_default", "f1_default", "roc_auc", "pr_auc_average_precision"]:
    if (metric, "A_all_features") in comparison.columns and (metric, "B_without_loan_grade") in comparison.columns:
        comparison[(metric, "delta_without_grade")] = (
            comparison[(metric, "B_without_loan_grade")] - comparison[(metric, "A_all_features")]
        )
delta_columns = [column for column in comparison.columns if column[1] == "delta_without_grade"]
display(comparison[delta_columns].style.format("{:+.3f}"))

grade_deltas = results_df.pivot(index=["model", "class_weight"], columns="scenario", values="pr_auc_average_precision")
grade_delta = grade_deltas["B_without_loan_grade"] - grade_deltas["A_all_features"]
largest_grade_change = grade_delta.abs().idxmax()
largest_change_value = grade_delta.loc[largest_grade_change]
print(f"Largest absolute PR-AUC change without loan_grade: {largest_grade_change} = {largest_change_value:+.3f}.")
print("Recommended candidate for further investigation (not a final winner):")
print(f"- {best_f1['configuration']}, because it has the strongest observed F1 balance on this fixed test evaluation. Validate it on a separate future/temporal or external dataset, examine calibration and subgroup fairness, and revisit feature availability before any decision-support use.")
print("- If minimizing missed defaults is the primary objective, compare the highest-recall model's precision and false-positive count with the operational cost of additional reviews.")
print("- This recommendation is provisional and is not an approval or denial rule.")

## 5. Save model pipelines and evaluation reports

Each fitted pipeline includes its scenario-specific train-fitted preprocessing and classifier. The evaluation CSV records all six algorithm/scenario pairs under both class-weight policies. A JSON report stores metrics, confusion matrices, parameters, package versions, and the fixed threshold. No transformed train/test datasets or approval decisions are saved.

In [ ]:
saved_model_paths = {}
for run_name, pipeline in fitted_pipelines.items():
    model_path = models_dir / f"credit_risk_{run_name}.joblib"
    joblib.dump(pipeline, model_path)
    saved_model_paths[run_name] = str(model_path.relative_to(project_root))

metrics_path = reports_dir / "credit_risk_model_evaluation.csv"
results_export = results_df.drop(columns=["confusion_matrix", "warnings"])
results_export.to_csv(metrics_path, index=False)
report_path = reports_dir / "credit_risk_model_evaluation.json"
report = {
    "objective": "Default-risk probability estimation for research and decision support; not an automated loan decision.",
    "source_split": {
        "train_file": str(train_path.relative_to(project_root)),
        "test_file": str(test_path.relative_to(project_root)),
        "random_state": config["split"]["random_state"],
        "stratified_by": target_column,
        "train_rows": int(len(X_train)),
        "test_rows": int(len(X_test)),
    },
    "threshold": 0.50,
    "threshold_tuning": "none; fixed at 0.50",
    "model_selection": "Test metrics are reported for comparison only; no tuning or selection used the test set.",
    "scenarios": {key: value for key, value in scenarios.items()},
    "class_weight_policies": ["default", "balanced"],
    "versions": {"python": sys.version.split()[0], "pandas": pd.__version__, "scikit_learn": sklearn.__version__},
    "feature_importance": "Impurity-based; descriptive, not causal.",
    "metrics": results,
    "saved_models": saved_model_paths,
}
report_path.write_text(json.dumps(report, indent=2), encoding="utf-8")

assert len(saved_model_paths) == 12
assert metrics_path.is_file() and report_path.is_file()
for run_name, path_string in saved_model_paths.items():
    assert (project_root / path_string).is_file(), run_name
print(f"Saved {len(saved_model_paths)} fitted model pipelines under {models_dir}.")
print(f"Metrics table: {metrics_path}")
print(f"Detailed evaluation report: {report_path}")
print("All reported scores use the persisted test split at a fixed 0.50 threshold. No hyperparameter search, threshold tuning, SMOTE, or automated approval decision was performed.")